# Multiprocessing
The multiprocessing API provides a suite of concurrency primitives for synchronizing and coordinating processes, as process-basd counterparts to the threading concurrency primitives. This includes `Lock`, `RLock`, `Semaphore`, `Event`, `Condition`, `Barrier`.

Process-safe queues are provided in `Queue`, `SimpleQueue` and so on that mimic the thread-safe queues provided in the `queue` module.

Provides more capabilities, focused on Inter-Process Communication (IPC), the manner in which data is transmitted between processes.

To process-safe versions of queues, `Connection` are provided that permit connection between processes both on the same system and across systems.

`Manager` API that creates a server process for managing centralised versions of Python objects.

In [2]:
from time import sleep
from random import random
from multiprocessing import (
    Process,
    current_process,
    parent_process,
    active_children,
    Lock,
    Semaphore,
    Event,
    Condition,
    Barrier,
    set_start_method,
    Value,
    Pipe,
    Queue,
    Manager,
    Pool
) 

## Create and Start a Child
**1. Main process.**

* *Main Process*: Default process created to execute a Python program, has the name `MainProcess`.
* *Main Thread*: Default thread created by a main process in a Python program, has the name `MainThread`.

Process started to run our program. Main thread of the process executes the entry point of our program.

**2. Difference between parent and child processes**
* *Parent Process*: Has one or more child processes. May have a parent process, e.g., may also be a child.
* *Child Process*: Has a parent process. May have its own child processes, e.g., may also be a parent.

A child process may inherit global variables from the parent process.

**3. The life cycle of Python processes including each step and their transitions.**

Three steps of its life-cycle: a new process, a running process, and a terminated process. While running, the process may be executing code or may be blocked, waiting on something such as another process or an external resource.

A process cannot exit normally until:
* All non-daemon threads have terminated, including the main thread.
* All non-daemon child processes have terminated, including the main process.

**4. Protect the entry point of the program and add freeze support.**
```python
if __name__ == "__main__":
    ...
```
Protecting the entry point voids a `RuntimeError` when creating a child process using the `spawn` start method, the default on Windows and MacOS.

A good practice to add freeze support as the first line of a Python program that uses the `multiprocessing` module. Freezing a Python program is a process that transforms the Python code into C code for packaging and distribution. Creating a process in a frozen application will result in a `RuntimeError`.
```python
freeze_support()
```

Protecting the entry point and adding freeze support together are referred to as the *main module* idiom when using `multiprocessing`.

**5. Run a function in a child process.**

1. Create an instance of the `Process` class.
2. Specify the name of the function via the `target` argument.
3. Call the `start()` method. 

In [ ]:
# custom function to be executed in a child process
def task():
    # block for a moment
    sleep(1)
    # report a message
    print("This is from another process", flush=True)
    
# protect the entry point 
if __name__ == "__main__":
    # create a new process instance
    process = Process(target=task)
    # start executing the function in the process
    process.start()
    # wait for the process to finish
    print("Waiting for the process...")
    process.join()

**6. Extend the `Process` class to run custom code in a child process.**

In [ ]:
# custom process class
class CustomProcess(Process):
    # override the run function
    def run(self):
        # block for a moment
        sleep(1)
        # report a message
        print("This is another process", flush=True)
        
# protect the entry point 
if __name__ == "__main__":
    # create the process
    process = CustomProcess()
    # start the process
    process.start()
    # wait for the process to finish
    print("Waiting for the process to finish.")
    process.join()

## Configuring and Interacting with Processes

**1. Configure the name of a process and whether it is a daemon.**

Two properties of a process that can be configured, they are the name of the process and whether the process is a daemon or not.

Processes can be configured to be *daemon* or *daemonic*, that is, they can configured as background process. A parent process can only exit once all non-daemon child processes have exited. This means tha daemon child processes can run in the background and do not prevent the main process of a Python program from exiting when the main parts of a program have finished.
```python
if __name__ == "__main__":
    process = Process(name="MyProcess", daemon=True)
    # report a process name
    print(process.name)
    # report if the process is a daemon
    print(process.daemon)
```

**2. Query the status of a process.**
* Process identifier (PID)

In [ ]:
# protect the entry point
if __name__ == "__main__":
    # create the process 
    process = Process()
    # report the process identifier. Confirms that it does not have a native PID before it was started
    print(process.pid)
    # start the process
    process.start()
    # report the process identifier
    print(process.pid)

* Whether the process is still running (or not). Alive or dead. An alive process means that the `run()` method of the `Process` instance is currently running.

In [ ]:
# protect the entry point
if __name__ == "__main__":
    # create the process
    process = Process()
    # report the process is alive
    print(process.is_alive())

* Exit code of the process (if terminated). A child process will have an exit code once it has terminated. Indication of whether processes completed successfully or not, and if not, the type of error that occurred that caused the termination. Common exit codes include: 0 for a normal exit and 1 for an error or failure of some kind.

In [ ]:
# custom function to be executed in a child process
def task():
    # block for a moment
    sleep(1)
    
# protect the entry point
if __name__ == "__main__":
    # create a process
    process = Process(target=task)
    # report the exit status
    print(process.exitcode)
    # start the process
    process.start()
    # report the exit status
    print(process.exitcode)
    # wait for the process to finish
    process.join()
    # report the exit status
    print(process.exitcode)

**3. Terminate and kill processes.**

A process may also be forcefully terminated or killed from another process. This involves raising a signal in the target process.
```python
process.terminate()
```

**4. Get access to the current, parent and child processes.**

In [ ]:
# protect the entry point
if __name__ == "__main__":
    # get the current process
    process = current_process()
    # report details
    print(process)

In [ ]:
# protect the entry point
if __name__ == "__main__":
    # get the current process
    process = parent_process()
    # report details
    print(process)
    
# The function returns None, as expected as the main process does not have a parent process.

List of all active child processes for a parent process. That are actually running.

In [ ]:
# custom function to be executed in a child process
def task():
    # block for a moment
    sleep(1)
    
# protect the entry point
if __name__ == "__main__":
    # create a number of child processes
    processes = [Process(target=task) for _ in range(5)]
    # start the child processes
    for process in processes:
        process.start()
    # get a list of all active child processes
    children = active_children()
    # report a count of active children
    print(f"Active Children Count: {len(children)}")
    # report each in turn
    for child in children:
        print(child)

**5. Configure the start method and use a multiprocessing context.**

A start method is the technique used to start child processes in Python. Three start methods:
* `spawn`: start a new Python process.
* `fork`: copy a Python process from an existing process.
* `forkserver`: new process from which future forked processes will be copied.

Windows and MacOS use `spawn`, whereas Linux uses `fork`. Windows does not support `fork` or `forkserver`.

```python
# get supported start methods
methods = get_all_supported_methods()
```
```python
# get the current start method
method = get_start_method()
```
```python
# set the start method
set_start_method("spawn")
```

It is best practice, and required on most platforms that the start method be set first.
```python
# protect the entry point
if __name__ == "__main__":
    # set the start method
    set_start_method("spawn")
```
We can use different start methods throughout our program by creating processes from different multiprocessing contexts.  

## Synchronize and Coordinate Processes

**1. Protect critical sections from race conditions with mutex locks.**

A mutual exclusion lock or mutex lock is a concurrency primitive intended to prevent a race condition.

A race condition is a concurrency failure case when two processes (or threads) run the same code and access or update the same resource (e.g., data variables, stream, etc.) leaving the resource in an unknown and inconsistent state.

Race conditions often result in unexpected behaviour of a program and/or corrupt data.

These sensitive parts of code that can be executed by multiple processes concurrently and may result in race conditions are called critical sections. A critical section may refer to a single block of code, but it also refers to multiple accesses to the same data variable or resource from multiple functions.

Only one process can have the lock at any time. If a process does not release an acquired lock, it cannot be acquired again.

The process attempting to acquire the lock will block until the lock is acquired, such as if another process currently holds the lock then releases it.

In [ ]:
# custom function to be executed in a child process
def task(shared_lock, ident, value):
    # acquire the lock
    with shared_lock:
        # report a message
        print(f">{ident} got lock, sleeping {value}", flush=True)
        # block for a fraction of a second
        sleep(value)
        
# protect the entry point 
if __name__ == "__main__":
    # create the shared mutex lock
    lock = Lock()
    # create a number of processes with different args
    processes = [Process(target=task, args=(lock, i, random())) for i in range(10)]
    # start the processes
    for process in processes:
        process.start()
    # wait for all processes to finish
    for process in processes:
        process.join()

Only one process can acquire the lock at a time and once they do, they report a message including their id and how long they will sleep. The process then blocks for a fraction of a second before releasing the lock.

**2. Limit access to be a protected resource with a semaphore.**

A semaphore is a concurrency primitive that allows a limit on the number of processes that can acquire a lock protecting a critical section or resource.

It is an extension of a mutual exclusion (mutex) lock that adds a count for the number of processes that can acquire the lock before additional processes will block. Once full, new processes can only acquire access on the semaphore once an existing process holding the semaphore releases access.

When a semaphore is created, the upper limit on the counter is set. If it is set to be 1, then the semaphore will operate like a mutex lock.

The semaphore can be acquired by calling the `acquire()` method. By default, it is a blocking call, which means that the calling process will block untill access becomes available on the semaphore. Once acquired, the semaphore can be released again by calling the `release()` method. 

In [ ]:
# custom function to be executed in a child process 
def task(shared_semaphore, ident):
    # attempt to acquire the semaphore
    with shared_semaphore:
        # generate a random value between 0 and 1
        val = random()
        # block for a fraction of a second
        sleep(val)
        # report result
        print(f"Process {ident} got {val}", flush=True)
        
# protect the entry point
if __name__ == "__main__":
    # create a shared semaphore
    semaphore = Semaphore(2)
    # create processes
    processes = [Process(target=task, args=(semaphore, i)) for i in range(10)]
    # start child processes
    for process in processes:
        process.start()
    # wait for child processes to finish
    for process in processes:
        process.join()

All ten processes attempt to acquire the semaphore, but only two processes are granted access at a time.

**3. Signal between process using an event.**

An event is a process-safe boolean flag that can be used to signal between two or more processes.

Processes sharing the event instance can check if the event is set, set the event, clear the event (make it not set), or wait for the event to be set.

The `Event` provides an easy way to share a boolean variable between processes that can act as a trigger for an action.

In [ ]:
# custom function to be executed in a child process
def task(shared_event, number):
    # wait for the event to be set
    print(f"Process {number} waiting ...", flush=True)
    shared_event.wait()
    # begin processing, generate a random number
    value = random()
    # block for a fraction of a second
    sleep(value)
    # report a message
    print(f"Process {number} got {value}", flush=True)
    
# protect the entry point
if __name__ == "__main__":
    # create a shared event object
    event = Event()
    # create a suit of processes
    processes = [Process(target=task, args=(event, i)) for i in range(5)]
    # start all processes
    for process in processes:
        process.start()
    # block for a moment
    print("Main process blocking...")
    sleep(2)
    # trigger all child processes
    event.set()
    # wait for all child processes to terminate
    for process in processes:
        process.join()

Running the example first creates and starts five child processes.

Each child process waits on the event before it starts its works, reporting a message that it is waiting.

The main process blocks for a moment, allowing all child processes to begin and start waiting on the event.

The main process then sets the event. This triggers all five child processes that perform their simulated work and report a message.

**4. Coordinate action with wait and notify using a condition variable.**

A condition variable (also called a monitor) allows multiple processes to wait and be notified about some result.

A condition can be acquired by a process after which it can wait to be notified by another process that something has changed. While waiting, the process is blocked and releases the lock on the condition for other processes to acquire.

Another process can then acquire the condition, make a change in the program, and notify one, all, or subset of processes waiting on the condition that something has changed.

The waiting process can then wake-up, re-acquire the condition, perform checks on any changed state and perform required actions.

The `wait()` method will wait forever until notified by default. We can also pass a `timeout` argument which will allow the process to stop blocking after a time limit in seconds.

We can notify a single waiting process via the `notify()` method. We can notify all processes waiting on the condition via the `notify_all()` method.

In [ ]:
# custom function to be executed in a child process
def task(shared_condition):
    # block for a moment
    sleep(1)
    # notify a waiting process that the work is done
    print(f"Child sending notification ...", flush=True)
    with shared_condition:
        shared_condition.notify()
        
# protect the entry point
if __name__ == "__main__":
    # create a condition
    condition = Condition()
    # acquire the condition
    print("Main process waiting for data ...")
    with condition:
        # create a nee process to execute the task
        worker = Process(target=task, args=(condition, ))
        # start the new child process
        worker.start()
        # wait to be notified by the child process
        condition.wait()
    # we know the data is ready
    print("Main process all done")

**5. Coordinate multiple processes at one point using a barrier.**

A barrier is a synchronization primitive.

It allows multiple processes to wait on the same barrier object instance (e.g., at the same point in code) until a predefined fixed number of processes arrive (e.g., the barrier is full), after which all processes are then notified and released to continue their execution.

Internally, a barrier maintains a count of the number of processes waiting on the barrier and a configured maximum number of parties (processes) that are expected. Once the expected number of parties reaches the pre-defined maximum, all waiting processes are notified.

This provides a useful mechanism to coordinate actions between multiple processes.

Specify the number of parties (processes) that must arrive before the barrier will be lifted.
```python
# configure a barrier with a action
barrier = Barrier(10, action=my_function)
```

In [ ]:
# custom function to be executed in a child process
def task(shared_barrier, ident):
    # generate a unique value between 0 and 10
    value = random() * 10
    # block for a moment
    sleep(value)
    # report result
    print(f"Process {ident} got: {value}", flush=True)
    # wait for all other processes to complete
    shared_barrier.wait()
    
# protect the entry point
if __name__ == "__main__":
    # create a barrier for (5 workers + 1 main process)
    barrier = Barrier(5 + 1)
    # create the worker processes
    workers = [Process(target=task, args=(barrier, i)) for i in range(5)]
    # start the worker processes
    for worker in workers:
        # start process
        worker.start()
    # wait for all worker processes to finish
    print("Main process waiting on all results...")
    barrier.wait()
    # report once all processes are done
    print("All processes have their results")

## Share Data Between Processes

## Run Tasks with Reusable Workers in Pools

## Share Centralized Objects with Managers